In [ ]:
from scripts.tokenizer import train_bpe_tokenizer
from scripts.vectorizer import Vectorizer
from scripts.vocabulary import Vocabulary
import pandas as pd
import json
import os
import sys
import json
from pathlib import Path
from dotenv import load_dotenv

In [2]:
if sys.platform == 'linux':
    load_dotenv(dotenv_path=(Path('.')/'.env.linux'))
elif sys.platform == 'win32':
    load_dotenv(dotenv_path=(Path('.')/'.env.win'))
else:
    raise ValueError('Ваша операционная система не поддерживается!')

DATASET_VERSION = os.getenv('DATASET_VERSION', '2.16') # Допустимые занчения: 2.3; 2.16 | В версии 2.3 меньше тренировочных примеров, по сравнению с 2.16. Точность на тестовой выборке практически не меняется
DATASET_PATH = os.getenv('DATASET_PATH')
CORPUS_TEXTS_FILEPATH = os.getenv('CORPUS_TEXTS_FILEPATH')

In [ ]:
UNK_TOKEN = '<UNK>'
MASK_TOKEN = '<MASK>'
PAD_TOKEN = '<PAD>'
BOS_TOKEN='<BOS>'
EOS_TOKEN = '<EOS>'
ADD_BOS_EOS_TOKENS = False

WORD_REPRESENTATION = 'tokens' # tokens; letters; both  Уровень представления слова (токены, буквы, токены + буквы)

VOCABULARY_SIZE = 10000
MIN_FRECQUENCY_PAIR = 80

MAX_WORDS_COUNT = 0
MAX_SUBTOKENS_COUNT = 0
MAX_LETTERS_COUNT = 12

In [4]:
def find_max_words_source_len(dataframe:pd.DataFrame)->int:
    '''Возвращает максимальную длину НЕ ТОКЕНИЗИРОВАННОЙ входной последовательности в датафрейме'''
    max_words_tokens = 0
    for i in range(len(dataframe)):
        max_words_tokens = max(len(dataframe.loc[i, 'source_words']), max_words_tokens)
    return max_words_tokens

In [5]:
def find_max_tokens_source_len(dataframe:pd.DataFrame, tokenizer)->int:
    '''Возвращает максимальную длину ТОКЕНИЗИРОВАННОЙ входной последовательности в датафрейме'''
    max_source_tokens = 0
    for i in range(len(dataframe)):
        max_source_tokens = max(len(tokenizer.encode(dataframe.loc[i, 'source_text']).tokens), max_source_tokens)
    return max_source_tokens

In [6]:
def find_max_subtokens_cnt(dataframe:pd.DataFrame)->int:
    '''Возвращает максимальное количество субтокенов слов в датафрейме'''
    max_subtokens_cnt = 0
    for row in range(len(dataframe)):
        for token_lst in dataframe.loc[row, 'tokens']:
            max_subtokens_cnt =  max(max_subtokens_cnt, len(token_lst))
    return max_subtokens_cnt

In [7]:
def find_max_letters_cnt(dataframe:pd.DataFrame)->int:
    '''Возвращает максимальное количество букв слов в датафрейме'''
    max_letters_cnt = 0
    for i in range(len(dataframe)):
        row_text_list = dataframe.loc[i, 'source_words']
        for word in row_text_list:
            max_letters_cnt = max(max_letters_cnt, len(word))
    return max_letters_cnt

In [8]:
def tokenize_dataset(dataframe:pd.DataFrame, tokenizer, return_length=False):
    dataframe['tokens'] = [[] for _ in range(len(dataframe))]
    if return_length:
        # dataframe['length'] = [0] * len(dataframe)
        dataframe['length'] = 0
    for row in range(len(dataframe)):
        for word in dataframe.loc[row, 'source_words']:
            tokens = tokenizer.encode(word).tokens
            dataframe.loc[row, 'tokens'].append(tokens)
            if return_length:
                 dataframe.loc[row, 'length'] += len(tokens)
    return dataframe

In [9]:
train_df = pd.read_parquet(os.path.join(DATASET_PATH, 'ru_syntagrus-ud-train.parquet'))
validation_df = pd.read_parquet(os.path.join(DATASET_PATH, 'ru_syntagrus-ud-dev.parquet'))
test_df = pd.read_parquet(os.path.join(DATASET_PATH, 'ru_syntagrus-ud-test.parquet'))

In [10]:
tokenizer = train_bpe_tokenizer([CORPUS_TEXTS_FILEPATH], VOCABULARY_SIZE, MIN_FRECQUENCY_PAIR, unk_token=UNK_TOKEN, pad_token=PAD_TOKEN)
print(len(tokenizer.get_vocab()))

train_df = tokenize_dataset(train_df, tokenizer)
test_df = tokenize_dataset(test_df, tokenizer)
validation_df = tokenize_dataset(validation_df, tokenizer)

6212


In [ ]:
MAX_WORDS_COUNT = max(find_max_words_source_len(test_df), find_max_words_source_len(validation_df))
MAX_SUBTOKENS_COUNT = max(find_max_subtokens_cnt(test_df), find_max_subtokens_cnt(validation_df))
MAX_LETTERS_COUNT = max(find_max_letters_cnt(test_df), find_max_letters_cnt(validation_df))
if ADD_BOS_EOS_TOKENS:
    MAX_WORDS_COUNT += 2

print(MAX_WORDS_COUNT)
print(MAX_SUBTOKENS_COUNT)
print(MAX_LETTERS_COUNT)

144
16
30


In [12]:
# Оставляем в обучающей выборке только строки длины не большей, чем в тестовой выборке (удаляются всего 2 записи)
train_df = train_df.loc[(train_df['source_words'].apply(len) <= MAX_WORDS_COUNT)]
train_df = train_df.reset_index()

In [13]:
target_names = ['upos', 'head', 'deprel', 'Mood', 'NumType', 'VerbForm',
       'ExtPos', 'Reflex', 'Polarity', 'Typo', 'NameType', 'InflClass',
       'Person', 'Poss', 'Animacy', 'Degree', 'Foreign', 'Variant', 'Number',
       'Gender', 'NumForm', 'Aspect', 'Case', 'PronType', 'Tense', 'Abbr', 'Voice']
source_name = 'source_text'

In [ ]:
source_vocab = Vocabulary(bos_token=BOS_TOKEN, eos_token=EOS_TOKEN, pad_token=PAD_TOKEN, mask_token=MASK_TOKEN, unk_token=UNK_TOKEN, add_bos_eos_tokens=ADD_BOS_EOS_TOKENS)
target_vocabs = {target_name: Vocabulary(bos_token=BOS_TOKEN, eos_token=EOS_TOKEN, pad_token=PAD_TOKEN,\
                                         mask_token=MASK_TOKEN, unk_token=UNK_TOKEN, add_bos_eos_tokens=ADD_BOS_EOS_TOKENS) for target_name in target_names}

# Заполненеие словаря входных токенов и словаря таргет меток
for df in [train_df, validation_df, test_df]:
    for row in range(len(df)):
        for token_lst in df.loc[row, 'tokens']:
            source_vocab.add_tokens(token_lst)
        for target_name in target_names:
            target_vocabs[target_name].add_tokens(df.loc[row, target_name])
        
pad_idx = source_vocab.pad_idx
source_vocab_len = len(source_vocab)
trg_vocabs_len = {key:len(target_vocabs[key]) for key in target_names}

# Словарь для букв
letters_vocab = Vocabulary(bos_token=BOS_TOKEN, eos_token=EOS_TOKEN, pad_token=PAD_TOKEN, mask_token=MASK_TOKEN, unk_token=UNK_TOKEN, add_bos_eos_tokens=False)

for df in [train_df, validation_df, test_df]:
    for i in range(len(df)):
        for word in df.loc[i, 'source_words']:
            letters_vocab.add_tokens(list(word))

letters_vocab_len = len(letters_vocab)

In [ ]:
print(f'Длина словаря токенов = {len(source_vocab)}')
print(f'Длина словаря символов = {letters_vocab_len}')
for key in target_names:
    print(f'Длина словаря признака {key} = {len(target_vocabs[key])}')

Количество батчей = 543
Длина словаря токенов = 6192
Длина словаря символов = 167
Длина словаря признака upos = 21
Длина словаря признака head = 137
Длина словаря признака deprel = 53
Длина словаря признака Mood = 7
Длина словаря признака NumType = 8
Длина словаря признака VerbForm = 8
Длина словаря признака ExtPos = 16
Длина словаря признака Reflex = 5
Длина словаря признака Polarity = 5
Длина словаря признака Typo = 5
Длина словаря признака NameType = 13
Длина словаря признака InflClass = 5
Длина словаря признака Person = 7
Длина словаря признака Poss = 5
Длина словаря признака Animacy = 6
Длина словаря признака Degree = 7
Длина словаря признака Foreign = 5
Длина словаря признака Variant = 5
Длина словаря признака Number = 6
Длина словаря признака Gender = 7
Длина словаря признака NumForm = 8
Длина словаря признака Aspect = 6
Длина словаря признака Case = 12
Длина словаря признака PronType = 15
Длина словаря признака Tense = 7
Длина словаря признака Abbr = 5
Длина словаря признака Vo

In [16]:
vectorizer = Vectorizer(tokenizer, source_vocab, target_vocabs, letters_vocab, WORD_REPRESENTATION, pad_idx)

In [17]:
train_df['input_ids'] = None
for target_name in target_names:
    train_df[f'{target_name}_ids'] = None

for i in range(len(train_df)):
    train_df.at[i, 'input_ids'] = vectorizer.vectorize_tokens(MAX_SUBTOKENS_COUNT, MAX_WORDS_COUNT, train_df.loc[i, 'tokens'])
    trg_vectoized = vectorizer.vectorize_targets(train_df.loc[i], target_names, MAX_WORDS_COUNT, ADD_BOS_EOS_TOKENS)
    for target_name in target_names:
        train_df.at[i, f'{target_name}_ids'] = trg_vectoized[target_name]

validation_df['input_ids'] = None
for target_name in target_names:
    validation_df[f'{target_name}_ids'] = None

for i in range(len(validation_df)):
    validation_df.at[i, 'input_ids'] = vectorizer.vectorize_tokens(MAX_SUBTOKENS_COUNT, MAX_WORDS_COUNT, validation_df.loc[i, 'tokens'])
    trg_vectoized = vectorizer.vectorize_targets(validation_df.loc[i], target_names, MAX_WORDS_COUNT, ADD_BOS_EOS_TOKENS)
    for target_name in target_names:
        validation_df.at[i, f'{target_name}_ids'] = trg_vectoized[target_name]

test_df['input_ids'] = None
for target_name in target_names:
    test_df[f'{target_name}_ids'] = None

for i in range(len(test_df)):
    test_df.at[i, 'input_ids'] = vectorizer.vectorize_tokens(MAX_SUBTOKENS_COUNT, MAX_WORDS_COUNT, test_df.loc[i, 'tokens'])
    trg_vectoized = vectorizer.vectorize_targets(test_df.loc[i], target_names, MAX_WORDS_COUNT, ADD_BOS_EOS_TOKENS)
    for target_name in target_names:
        test_df.at[i, f'{target_name}_ids'] = trg_vectoized[target_name]

In [18]:
train_df = train_df.drop(columns=[*[target_name for target_name in target_names], 'index', 'lemmas', 'xpos', 'feats', 'misc', 'source_text'])
validation_df = validation_df.drop(columns=[*[target_name for target_name in target_names], 'lemmas', 'xpos', 'feats', 'misc', 'source_text'])
test_df = test_df.drop(columns=[*[target_name for target_name in target_names], 'lemmas', 'xpos', 'feats', 'misc', 'source_text'])

In [19]:
with open('data/vocabs_configuration.json', 'w', encoding='utf-8') as file:
    json.dump({'MAX_WORDS_COUNT' : MAX_WORDS_COUNT,
    'MAX_SUBTOKENS_COUNT' : MAX_SUBTOKENS_COUNT,
    'MAX_LETTERS_COUNT' : MAX_LETTERS_COUNT,
    'SOURCE_VOCAB_LEN' :  source_vocab_len,
    'LETTERS_VOCAB_LEN' : letters_vocab_len,
    'TRG_VOCABS_LEN' : trg_vocabs_len,
    'PAD_IDX' : pad_idx}, file, indent=4, ensure_ascii=False)

train_df.to_parquet(os.path.join(DATASET_PATH, 'prepared_ru_syntagrus-ud-train.parquet'), engine="fastparquet", index=False)
test_df.to_parquet(os.path.join(DATASET_PATH, 'prepared_ru_syntagrus-ud-test.parquet'), engine="fastparquet", index=False)
validation_df.to_parquet(os.path.join(DATASET_PATH, 'prepared_ru_syntagrus-ud-dev.parquet'), engine="fastparquet", index=False)